# Social Content to Donation Impact Optimization

## 1. Problem Framing
- **Business question:** Which social post characteristics drive donation referrals and donation value?
- **Predictive goal:** estimate expected referrals/value before posting.
- **Explanatory goal:** quantify associations between content choices and donation outcomes.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

DATA = Path('lighthouse_csv_v7')
posts = pd.read_csv(DATA / 'social_media_posts.csv', parse_dates=['created_at'])
posts = posts.sort_values('created_at').copy()

posts['target_referrals'] = posts['donation_referrals'].fillna(0)
posts['target_value'] = posts['estimated_donation_value_php'].fillna(0)

# Leakage checklist:
# Use only fields known at publish time. Exclude post-performance fields like reach/likes/views.
feature_cols = [
    'platform','day_of_week','post_hour','post_type','media_type','num_hashtags',
    'mentions_count','has_call_to_action','call_to_action_type','content_topic','sentiment_tone',
    'caption_length','is_boosted','boost_budget_php'
]

df = posts[['created_at'] + feature_cols + ['target_referrals','target_value']].copy()

cut = int(len(df) * 0.8)
train_df = df.iloc[:cut].copy()
test_df = df.iloc[cut:].copy()

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
X = df[feature_cols]
y_ref = df['target_referrals']
y_val = df['target_value']

num_cols = X_train.select_dtypes(include=['number','bool']).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

def run_regression(y_col, label, log_target=False):
    y_train = train_df[y_col]
    y_test = test_df[y_col]

    baseline = np.repeat(float(y_train.mean()), len(y_test))
    print(label, 'Baseline_Mean', 'MAE', round(mean_absolute_error(y_test, baseline), 2), 'R2', round(r2_score(y_test, baseline), 3))

    exp_model = Pipeline([('pre', pre), ('reg', LinearRegression())])
    pred_model = Pipeline([('pre', pre), ('reg', RandomForestRegressor(n_estimators=400, random_state=42))])

    if log_target:
        y_train_fit = np.log1p(y_train)
    else:
        y_train_fit = y_train

    for name, model in [('Explanatory_Linear', exp_model), ('Predictive_RF', pred_model)]:
        model.fit(X_train, y_train_fit)
        p = model.predict(X_test)
        if log_target:
            p = np.expm1(p)
        print(label, name + ('_LogTarget' if log_target else ''), 'MAE', round(mean_absolute_error(y_test, p), 2), 'R2', round(r2_score(y_test, p), 3))

run_regression('target_referrals', 'DonationReferrals', log_target=False)
run_regression('target_value', 'DonationValuePHP', log_target=True)


## 4. Evaluation & Interpretation
- MAE helps interpret expected absolute error in referrals/value.
- R2 indicates how much variation in outcomes is captured by content and distribution features.

## 5. Causal and Relationship Analysis
- CTA type, content topic, boost budget, and timing often show strong associations with fundraising outcomes.
- Because this is observational, effects should be validated with experiments (A/B tests) before causal claims.

## 6. Deployment Notes
- Endpoint: `POST /api/ml/social-post-impact`.
- UI: pre-publish planner estimates expected referrals and donation value for draft posts.

In [ ]:
# Model selection for social content impact (referrals + donation value)
from sklearn.model_selection import KFold, cross_validate
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

candidate_regressors = {
    'LinearRegression': LinearRegression(),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=300, random_state=42),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=42)
}

cv_splits = min(5, len(X))
cv_splits = max(2, cv_splits)
cv = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
scoring = {'mae': 'neg_mean_absolute_error', 'r2': 'r2'}

def select_best_regressor(y, label):
    rows = []
    for model_name, estimator in candidate_regressors.items():
        pipe = Pipeline([('pre', pre), ('reg', estimator)])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1, error_score='raise')
        rows.append({
            'model': model_name,
            'cv_folds': cv_splits,
            'mae_mean': -scores['test_mae'].mean(),
            'r2_mean': scores['test_r2'].mean()
        })
    out = pd.DataFrame(rows).sort_values('mae_mean', ascending=True)
    print(f'Model selection results ({label}):')
    print(out.to_string(index=False))
    print('-' * 70)
    return out

referral_model_selection = select_best_regressor(y_ref, 'Donation Referrals')
value_model_selection = select_best_regressor(y_val, 'Donation Value PHP')